<a href="https://colab.research.google.com/github/heisenberg304/heart-disease-ml-classification/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ORDEN DE ACCION DE LOS MODULOS

#1. MODULO ONE

#2. MODULO TWO

#3. MODULO THREE

#4. MODULO MAIN

In [18]:

#MAIN MODULO
from model_1 import model_binary, predict_modelo_1, x_test, todo as model_binary_1_performance, matrix as matrix_model_1    #binaria
from model_2 import model_multiclass, predict_modelo_2, todo as model_binary_2_performance, matrix as matrix_model_2        #multiclase
from data_engineering import data, FEATURES, split_dataset

import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

#x_test puede ser reemplazado por cualquier dataset de paccientes nuevos
new_patients = x_test

#pacientes totalmente nuevos, solo features
def pred_binaria(new_patients):
  x = new_patients[FEATURES]   #como es nuevo no tiene la columna target ni nada solo los Features, tambien sirve por si en los datos nuevos hay otras columnas innecesarias como id o name
  pred_1 = predict_modelo_1(model_binary, x)
  return pred_1
pred_1 = pred_binaria(new_patients)

def attach_predictions(new_patients, pred):
  results = new_patients.copy()
  results["prediction"] = pred
  return results
patients_pred_1 = attach_predictions(new_patients, pred_1)

def h(patients_pred_1):
  patients_pred_1 = patients_pred_1.copy()
  patients_pred = patients_pred_1[patients_pred_1["prediction"] > 0]
  sanos = patients_pred_1[patients_pred_1["prediction"] == 0]
  return patients_pred, sanos
enfermos, sanos = h(patients_pred_1)

def pred_multiclass(enfermos):
  x = enfermos[FEATURES]
  pred_2 = predict_modelo_2(model_multiclass, x)
  return pred_2
pred_2 = pred_multiclass(enfermos)

def clasificacion_completa(pred_2, enfermos, sanos):
  patients_pred_2 = attach_predictions(enfermos, pred_2)
  completa = pd.concat([patients_pred_2, sanos], axis=0)
  return completa
completa = clasificacion_completa(pred_2, enfermos, sanos)

print(completa)

#imprimir rendimiento de modelos - quitar '#'
#print(f"{model_binary_1_performance} matrix confusion: \n{matrix_model_1} \n ----------------------------------------------------------- \n {model_binary_2_performance} matrix confusion: \n {matrix_model_2}")

     age  trestbps   chol    fbs  thalch  exang  oldpeak  Female   Male  \
35    42     140.0  226.0  False   178.0  False      0.0   False   True   
777   60     130.0  316.0   True   140.0  False      0.5   False   True   
79    58     150.0  270.0  False   111.0   True      0.8   False   True   
133   51     140.0  261.0  False   186.0   True      0.0   False   True   
681   61     105.0    0.0  False   110.0   True      1.5   False   True   
..   ...       ...    ...    ...     ...    ...      ...     ...    ...   
309   32     110.0  225.0  False   184.0  False      0.0   False   True   
493   34     140.0  156.0  False   180.0  False      0.0   False   True   
352   41     120.0  295.0  False   170.0  False      0.0   False   True   
422   51     150.0  200.0  False   120.0  False      0.5    True  False   
4     41     130.0  204.0  False   172.0  False      1.4    True  False   

     asymptomatic  atypical angina  non-anginal  typical angina  \
35           True            Fal

#Motivo del por que modelo de cascade de 2 etapas:

Estos modelos existian de manera independientes como proyectos.

Mientras que uno era muy preciso en catalogar enfermos, recall de 93-95, pero no clasificaba la cantidad de posibles enfermedades cardiacas. Entiendo que este modelo es el mas importante de los 2, ya que si este falla, el paciente sera clasificado como sano y no sera diagnostica, y corre peligro, tambien por eso modifique su threshold, para disminuir lo mas pasible el numero de FN, osea de los enfermos que pasa por alto.

---
En cambio el otro modelo multiclass si podia clasificar el numero de posibles enfermedades cardiacas, pero habia perdido mucho rendimiento, se le pasaba el 60% de los enfermos, algo fatal para uso medico, especialmente si lo comparanos al otro modelo binario que solo se le escapaban del 5-7%.

Lo primero que pense es que era el ruido que generaba el que hayan personas sanas en el entrenamiento del modelo multiclass, y que seria mejor separar sus funcionalidades, recien me entero que se llama modelo de cascada, yo solo pense en lo que podia funcionar y listo.

In [1]:
#MODULO ONE
%%writefile data_engineering.py

#Cargar librerias
import pandas as pd
from sklearn.model_selection import train_test_split


#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")

#===================================================================================================================================================================================================

#MANIPULAR CSV
  #rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
def handle_missing_values(data):
  data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
  data["chol"].fillna(data["chol"].median(), inplace=True)
  data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
  data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
  data["thalch"].fillna(data["thalch"].median(), inplace=True)
  data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)
  data["exang"].fillna(data["exang"].mode()[0], inplace=True)

  return data

data = handle_missing_values(data)

def encode_categorical_features(data):
    #encoding: sex, cp, restecg, num (target)
  fe_ma = pd.get_dummies(data["sex"])
  CP = pd.get_dummies(data["cp"])
  RESTECG = pd.get_dummies(data["restecg"])
  return fe_ma, CP, RESTECG

fe_ma, CP, RESTECG = encode_categorical_features(data)


def feature_selection(data, fe_ma, CP, RESTECG):
    #union de datasets
  data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
    #borrar columnas
  data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)
  return data

data = feature_selection(data, fe_ma, CP, RESTECG)

#===================================================================================================================================================================================================

FEATURES = ['age','trestbps','chol','fbs','thalch','exang',
            'oldpeak','Female','Male','asymptomatic',
            'atypical angina','non-anginal','typical angina',
            'lv hypertrophy','normal','st-t abnormality']

##############################################################

#modelo 1
def prepare_binary_dataset(data):
  #copiar dataframe
  data = data.copy()
      #target binario - separar enfermos y no enfermos
  data["num"] = data["num"].apply(lambda x: 1 if x > 0 else 0)
  # crear x , y
  x = data[FEATURES]
  y = data["num"]
  return x, y

##############################################################

#modelo 2
def prepare_multiclass_dataset(data):
  #copiar dataframe
  data = data.copy()
      #target multivariable - clasificación de severidad
  data = data[data["num"] > 0]
  # crear x , y
  x = data[FEATURES]
  y = data["num"]
  return x, y

##############################################################

def split_dataset(x, y):
  #split 80/20
  x_train, x_test, y_train, y_test = train_test_split(
      x,
      y,
      test_size = 0.2,
      random_state=55
  )
  return x_train, x_test, y_train, y_test


Writing data_engineering.py


In [2]:
#MODULO TWO
%%writefile model_1.py

#funciones de data_engineering
from data_engineering import split_dataset, prepare_binary_dataset, data

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


x, y = prepare_binary_dataset(data)
x_train, x_test, y_train, y_test = split_dataset(x, y)

def modelo_1():
  #crear modelo
  model = RandomForestClassifier(class_weight = "balanced",
                                n_estimators=200,
                                max_depth=7,
                                min_samples_leaf=6,
                                random_state=1,
                                max_features="sqrt")
  return model

model_binary = modelo_1()

def entrenar_modelo_1(model_binary, x_train, y_train):
  #entrenar modelo
  model_binary.fit(x_train, y_train)
  return model_binary

model_binary = entrenar_modelo_1(model_binary, x_train, y_train)

def predict_modelo_1(model_binary, x_test):
  #prediccion
  probs = model_binary.predict_proba(x_test)[:, 1]
  threshold = 0.34
  pred = (probs >= threshold).astype(int)
  return pred

pred = predict_modelo_1(model_binary, x_test)

def importances_modelo_1(model_binary):
  #Importancia de cada feature
  importances = model_binary.feature_importances_
  return importances

importances = importances_modelo_1(model_binary)

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)

#print de metricas
print(f"matrix -> {matrix}")
print(f"todo -> {todo}")
print(f"Importancia = {importances}")


Writing model_1.py



#BINARY MODEL PERFORMANCE

---
accuracy  -> 0.81

precision -> 0.77

recall    -> 0.93

f1 score  -> 0.84

---
Analisis:
El modelo logra un recall alto, lo cual indica que detecta correctamente la mayoría de pacientes con enfermedad.
Sin embargo, la precisión es menor, lo que implica mayor presencia de falsos positivos.
Este comportamiento es aceptable dado que en un contexto médico es preferible no dejar pasar casos enfermos.


---
Los 3 Importance mas relevantes
"asymptomatic" [0.22129586] El modelo detecta que la ausencia de síntomas típicos no implica ausencia de enfermedad, lo cual es clínicamente coherente.
"oldpeak"      [0.12007416] Es una variable médica directa relacionada con problemas cardíacos, por lo que tiene fuerte poder predictivo.
"exang"        [0.11460541] Indica que el corazón no responde bien al esfuerzo, lo que es un fuerte indicador clínico




In [3]:

#MODELO THREE
%%writefile model_2.py

#funciones de data_engineering
from data_engineering import split_dataset, prepare_multiclass_dataset, data

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

x, y = prepare_multiclass_dataset(data)
x_train, x_test, y_train, y_test = split_dataset(x, y)


def modelo_2():
  #crear modelo
  model = RandomForestClassifier(class_weight = "balanced",
                                n_estimators=200,
                                max_depth=9,
                                min_samples_leaf=6,
                                random_state=1,
                                max_features="sqrt")
  return model

model_multiclass = modelo_2()

def entrenar_modelo_2(model_multiclass, x_train, y_train):
  #entrenar modelo
  model_multiclass.fit(x_train, y_train)
  return model_multiclass

model_multiclass = entrenar_modelo_2(model_multiclass, x_train, y_train)

def predict_modelo_2(model_multiclass, x_test):
  #prediccion
  pred = model_multiclass.predict(x_test)
  return pred

pred = predict_modelo_2(model_multiclass, x_test)

def importances_modelo_2(model_multiclass):
  #Importancia de cada feature
  importances = model_multiclass.feature_importances_
  return importances

importances = importances_modelo_2(model_multiclass)

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)

#print de metricas
print(f"matrix -> {matrix}")
print(f"todo -> {todo}")
print(f"Importancia = {importances}")

data.isnull().sum()


Writing model_2.py


# MULTICLASS MODEL PERFORMANCE

analisis:

Este modelo se encarga catalogar la severidad, el numero de posibles enfermedades en el paciente, no es que no sea importante, pero no es desicivo como el anterior modelo binario, ademas al tener mas outputs, tiene menos precision, ya que tiene que catalogar dentro de una misma categoria, catalogar la severidad del enfermo dentro de los enfermos

Viendo precision, recall, f1, sabemos que lo que mejor se le da es cuando el paciente tiene una sola enfermedad (70%), y lo que peor se le da es cuando el paciente tiene 4 enfermedades (0.21), con un accuracy promedio de casi 0.5, podemos entender que el modelo cataloga bien el 50% de las veces, esto creo yo que es dada la naturaleza del dataset, ya que cada cuerpo humano es unico en como vive las enfermedades, y los contextos unicos de cada persona influyen en sus enfermedades, por eso catalogar enfermedades es una linea difusa y dificil de distinguir, no es una separacion clara y binaria como enfermo o no enfermo.